# NeXo v3.0 · Notebook 02 — CEM Score (LightGBM DART regression)

> **CRISP-DM Phase 4 — Modeling** for the CEM-score prediction model.
> Audience: thesis jury + Huawei supervisor. Defense window mid-Jun → mid-Jul 2026.

## What this notebook does

Trains a **LightGBM regression** model to predict the per-subscriber CEM score (∈[0,1]) computed by notebook 01.

## Input contract

- `curated/subscribers.parquet` — built by notebook 01 (CEM target + features)
- `curated/splits.json` — train/val/test indices + temporal-holdout months

## Output contract (DO NOT BREAK — consumed by ai-service)

- `models/cem_v3_lightgbm.joblib` — the fitted LightGBM model
- `models/cem_v3_feature_names.joblib` — ordered feature names (for inference validation)
- `models/cem_v3_model_card.md` — performance summary (R² / MAE / RMSE, random + temporal)
- Mirror to MinIO `curated/models/` for hot-reload

Consumed by `services/ai-service/model_cache.py` (mtime-based reload). Filenames immutable.

## What this notebook does NOT do

- No Optuna hyperparameter search (deferred — see `reports/02_explainer.md`)
- No SHAP global + per-area analysis (deferred)
- No isotonic calibration (deferred)
- No StratifiedKFold cross-validation — only single train/val/test split (deferred)
- No PR curve / threshold sweep (CEM is regression, not classification — but a binarized version for "low-CEM subscriber" intervention triggering would benefit)

## Pipeline diagram

```
curated/subscribers.parquet ─┐
curated/splits.json ─────────┴─→ feature selection
                                       │
                                       ▼
                              LightGBM DART (256 leaves, depth=12, dart)
                                       │
                                       ▼
                              random-split metrics
                              temporal-holdout metrics
                                       │
                                       ▼
                              models/cem_v3_lightgbm.joblib
                                       │
                                       ▼
                              MinIO curated/models/
                                       │
                                       ▼
                              ai-service /infer/cem  (hot-reload via mtime)
```

## 0 · Reproducibility & Hyperparameter Constants (papermill-tagged)

**What this shows:** All hyperparameters and artifact filenames in ONE cell.
**What to look for:** `random_state=42` (model-level) was already set; this cell adds full notebook-level seeding (numpy + random + PYTHONHASHSEED) + names every hyperparameter for papermill override at retrain time.

In [ ]:
# --- Papermill parameters cell (TAGGED `parameters` in metadata) ---
# Consumed by `pb-model-retrain` playbook which papermill-overrides at runtime.

# --- Reproducibility ---
import os, random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# --- Hyperparameters (defaults from v3.0 production) ---
LGB_BOOSTING_TYPE       = 'dart'    # dropouts-meet-multiple-additive-regression-trees (Vinayak & Gilad-Bachrach 2015)
LGB_OBJECTIVE           = 'regression'
LGB_METRIC              = 'rmse'
LGB_NUM_LEAVES          = 256       # aggressive; DART tolerates more leaves due to dropout regularization
LGB_MAX_DEPTH           = 12        # cap on tree depth (interaction depth)
LGB_LEARNING_RATE       = 0.05      # standard shrinkage; lower = more trees needed
LGB_N_ESTIMATORS        = 600       # max boosting rounds (early stopping kicks in usually around 200-400)
LGB_MIN_CHILD_SAMPLES   = 20        # min leaf size (regularization)
LGB_REG_ALPHA           = 0.05      # L1 leaf-weight regularization
LGB_REG_LAMBDA          = 0.05      # L2 leaf-weight regularization
EARLY_STOP_ROUNDS       = 30        # patience on validation RMSE
LOG_EVAL_PERIOD         = 100

# --- Feature engineering policy ---
TOP_K_FEATURE_IMPORTANCE = 20       # how many features to plot in importance chart

# --- Artifact filenames (IMMUTABLE — consumed by ai-service/model_cache.py) ---
MODEL_FNAME             = 'cem_v3_lightgbm.joblib'
FEATURES_FNAME          = 'cem_v3_feature_names.joblib'
CARD_FNAME              = 'cem_v3_model_card.md'

print(f'SEED                  = {SEED}')
print(f'LGB_BOOSTING_TYPE     = {LGB_BOOSTING_TYPE!r}')
print(f'LGB_NUM_LEAVES        = {LGB_NUM_LEAVES}')
print(f'LGB_MAX_DEPTH         = {LGB_MAX_DEPTH}')
print(f'LGB_LEARNING_RATE     = {LGB_LEARNING_RATE}')
print(f'LGB_N_ESTIMATORS      = {LGB_N_ESTIMATORS}')
print(f'EARLY_STOP_ROUNDS     = {EARLY_STOP_ROUNDS}')
print(f'MODEL_FNAME           = {MODEL_FNAME!r}')

## 1 · Imports + MinIO client

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
from IPython.display import display

import io, json, os
from pathlib import Path
import boto3, joblib, numpy as np, pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt, seaborn as sns
from botocore.client import Config
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
sns.set_theme(style='whitegrid')
s3 = boto3.client('s3', endpoint_url=os.environ.get('S3_ENDPOINT','http://localhost:9000'),
                  aws_access_key_id='minio', aws_secret_access_key='minio_pw',
                  config=Config(signature_version='s3v4'))
print('lightgbm', lgb.__version__)

## 2 · Load curated/subscribers.parquet + splits.json

In [ ]:
subs = pd.read_parquet(io.BytesIO(s3.get_object(Bucket='curated', Key='subscribers.parquet')['Body'].read()))
splits = json.loads(s3.get_object(Bucket='curated', Key='splits.json')['Body'].read())
print(f'subscribers: {len(subs):,} rows  splits: {splits["meta"]["random_sizes"]}')

## 3 · Feature selection

Drop ID + target + metadata. Keep all numeric features.

In [ ]:
DROP = {'imsi','imsi_hash','cem_score_target','churn_risk_flag','rat_gap_score',
        'source_origin','source_file','_strata','month_year'}
feat_cols = [c for c in subs.columns if c not in DROP and pd.api.types.is_numeric_dtype(subs[c])]
print(f'features ({len(feat_cols)}):', feat_cols)
X = subs[feat_cols].fillna(0); y = subs['cem_score_target']

## 4 · Apply random split (train/val/test)

In [ ]:
tr_idx = splits['random']['train']; va_idx = splits['random']['val']; te_idx = splits['random']['test']
X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
X_va, y_va = X.iloc[va_idx], y.iloc[va_idx]
X_te, y_te = X.iloc[te_idx], y.iloc[te_idx]
print(f'train={len(X_tr):,} val={len(X_va):,} test={len(X_te):,}')

## 5 · Train LightGBM DART

DART config from `docs/architecture/ml-models.md`. Early stopping on validation RMSE.

In [ ]:
params = dict(boosting_type='dart', objective='regression', metric='rmse',
              num_leaves=256, max_depth=12, learning_rate=0.05, n_estimators=600,
              min_child_samples=20, reg_alpha=0.05, reg_lambda=0.05,
              random_state=42, verbose=-1)
model = lgb.LGBMRegressor(**params)
model.fit(X_tr, y_tr,
          eval_set=[(X_va, y_va)],
          callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False),
                     lgb.log_evaluation(period=100)])

## 6 · Test-set metrics (random split — primary)

In [ ]:
pred = model.predict(X_te)
m_random = {
    'R2': float(r2_score(y_te, pred)),
    'MAE': float(mean_absolute_error(y_te, pred)),
    'RMSE': float(np.sqrt(mean_squared_error(y_te, pred))),
}
print('Random split test:', json.dumps(m_random, indent=2))

## 7 · Temporal hold-out (production realism)

In [ ]:
ho = splits['temporal']['holdout_indices']
X_ho, y_ho = X.iloc[ho], y.iloc[ho]
pho = model.predict(X_ho)
m_temp = {
    'R2': float(r2_score(y_ho, pho)),
    'MAE': float(mean_absolute_error(y_ho, pho)),
    'RMSE': float(np.sqrt(mean_squared_error(y_ho, pho))),
    'holdout_size': int(len(ho)),
    'holdout_months': splits['temporal']['holdout_months'],
}
print('Temporal hold-out:', json.dumps(m_temp, indent=2))

## 8 · Feature importance plot

In [ ]:
imp = pd.Series(model.feature_importances_, index=feat_cols).sort_values(ascending=False).head(20)
fig, ax = plt.subplots(figsize=(8,7))
sns.barplot(x=imp.values, y=imp.index, palette='viridis', orient='h', ax=ax)
ax.set_title('CEM — top 20 features'); plt.tight_layout(); plt.show()

## 9 · Save model + push to MinIO curated/models/

In [ ]:
MODELS = Path('models'); MODELS.mkdir(exist_ok=True)
joblib.dump(model, MODELS/'cem_v3_lightgbm.joblib')
joblib.dump(feat_cols, MODELS/'cem_v3_feature_names.joblib')
card = (f'# CEM v3 Model Card\n\n'
        f'## Random split\n- R²={m_random["R2"]:.4f}\n- MAE={m_random["MAE"]:.4f}\n- RMSE={m_random["RMSE"]:.4f}\n\n'
        f'## Temporal hold-out\n- R²={m_temp["R2"]:.4f}\n- MAE={m_temp["MAE"]:.4f}\n- RMSE={m_temp["RMSE"]:.4f}\n'
        f'- months: {m_temp["holdout_months"]}\n')
(MODELS/'cem_v3_model_card.md').write_text(card); print(card)
for fn in ['cem_v3_lightgbm.joblib','cem_v3_feature_names.joblib','cem_v3_model_card.md']:
    s3.put_object(Bucket='curated', Key=f'models/{fn}', Body=(MODELS/fn).read_bytes())
    print(f'  ↑ curated/models/{fn}')

---
## Done · register the model

Copy `models/cem_v3_lightgbm.joblib` to `services/ai-service/models/` (or use `pb-retrain-model`
playbook from the dashboard which does this + hot-reload).